# P4 — Solar filament instance segmentation (YOLO11-seg baseline)

Competition: **Solar Filament Segmentation Challenge 2026** (IEEE BigData Cup 2026)

## Facts pinned down before writing code (do not re-derive these)
- Data mount: `/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/`
  (note the extra `competitions/` level — a hard-coded path without it raises FileNotFoundError)
- `train/` holds `train_images/` + one COCO json (48.6 MB); `test/` holds `test_images/` only.
- 887 jpeg in total, 2048x2048, **grayscale** (do not treat as RGB).
- Filenames encode capture time + instrument: `YYYYMMDDHHMMSSII` (e.g. `20160920230134Lh.jpeg`).
- **There is no `sample_submission.csv` in the data.** The real submission format is given on
  Overview -> Submission File: two columns `filament_id,segmentation_rle`, one row per predicted
  filament, `filament_id = <image_stem>_<n>`, `segmentation_rle` = COCO RLE **counts** string
  (no quotes, size fixed at 2048x2048, counts only).
- Leaderboard shows **mean Dice** on ~50% of the test set; the final rubric ranks by **PQ**.
- Set `Accelerator = GPU T4 x2` and `Internet = on` in the notebook settings; `No persistence`
  means ultralytics has to be reinstalled at the top of every session.

## Cell 1 — environment and paths

In [ ]:
# Competition: Solar Filament Segmentation Challenge 2026 -> YOLO11-seg baseline.
# Pipeline: install -> inspect COCO json -> export YOLO dataset (date-grouped split)
#           -> train -> predict -> RLE submission -> local Dice/PQ validation.

!pip install -q ultralytics pycocotools 2>&1 | tail -2

import os, json, glob, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
from pycocotools import mask as maskUtils

ROOT      = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
TRAIN_IMG = f'{ROOT}/train/train_images'
ANN_JSON  = f'{ROOT}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'
TEST_IMG  = f'{ROOT}/test/test_images'
WORK      = '/kaggle/working'

for p in (TRAIN_IMG, ANN_JSON, TEST_IMG):
    assert os.path.exists(p), f'missing: {p}'
print('paths ok')
print('train jpeg:', len(os.listdir(TRAIN_IMG)), '| test jpeg:', len(os.listdir(TEST_IMG)))

## Cell 2 — inspect the COCO annotation

Sanity numbers to look at before training:
- how many COCO image entries vs distinct `file_name` (the same physical frame can be annotated
  by several annotators, so entries > files is expected and allowed),
- how many filaments per image (a rough idea of the instance density),
- the segmentation field type (polygon list vs RLE).

In [36]:
coco = json.load(open(ANN_JSON))
imgs, anns = coco['images'], coco['annotations']
print('image entries :', len(imgs))
print('annotations   :', len(anns))
print('categories    :', [(c['id'], c['name']) for c in coco['categories']])
print('sample image  :', imgs[0])
print('sample ann    :', {k: str(v)[:90] for k, v in anns[0].items()})

fnames = [im['file_name'] for im in imgs]
print('distinct file_name          :', len(set(fnames)))
print('file_names annotated >1 time:', sum(v > 1 for v in Counter(fnames).values()))
print('mean annotations per image  :', round(len(anns) / max(len(imgs), 1), 2))
print('iscrowd values              :', set(a['iscrowd'] for a in anns))
print('segmentation field type     :', type(anns[0]['segmentation'][0]).__name__)

image entries : 1154
annotations   : 8199
categories    : [(1, 'Left'), (2, 'Right'), (3, 'Unidentifiable'), (4, 'Ambiguous')]
sample image  : {'license': 1, 'file_name': '20140609195854Bh.jpeg', 'url': 'https://gong2.nso.edu/HA/hag/201406/20140609/20140609195854Bh.jpg', 'height': 2048, 'width': 2048, 'date_captured': '2014-06-09 19:58:54', 'id': '040301-20140609195854Bh'}
sample ann    : {'segmentation': '[[480.9382, 850.955, 480.307, 852.0535, 478.2989, 852.451, 476.1202, 851.3087, 471.8813, 8', 'area': '750.0', 'iscrowd': '0', 'spine': '[388.7268, 805.455, 394.4037, 809.9707, 396.7261, 811.0029, 399.0485, 811.3899, 401.6289, ', 'image_id': '040301-20140609195854Bh', 'bbox': '[388.4121, 799.966, 92.52639999999997, 52.485000000000014]', 'category_id': '2', 'id': 'd7fd9cdb-3f34-4dfc-a1fa-c1fd25110a98'}
distinct file_name          : 707
file_names annotated >1 time: 296
mean annotations per image  : 7.1
iscrowd values              : {0}
segmentation field type     : list


## 
Cell 3 — export a YOLO segmentation dataset with a date-grouped split

**Why not a random split?** The Sun rotates slowly, so frames captured on neighbouring days are
almost identical. A random split would put "yesterday's same face" into validation and inflate the
score. We therefore split by calendar month.

**Why whole months, not "the last week of each month"?** From the month histogram, coverage is very
uneven (2011-2017: 3-19 images per month; 2018+: often 1-8, some months missing). Holding out a week
would leave some validation months with a single image, so whole months are held out instead.

`VAL_EVERY = 4` keeps every 4th month (in chronological order) for validation, spread across all
years. Files sharing one `file_name` always land in the same split, so annotator copies cannot leak.

Images are **symlinked** (not copied) under their COCO `image_id`, because several COCO entries share
one `file_name` and would otherwise overwrite each other's label file.

In [37]:
FNAME_RE = re.compile(r'^(\d{8})(\d{6})([A-Za-z]{2})$')

def date_key(file_name):
    """'20160920230134Lh.jpeg' -> '201609' (YYYYMM); None if the name does not match."""
    stem = os.path.splitext(file_name)[0]
    return stem[:6] if FNAME_RE.match(stem) else None

VAL_EVERY = 4   # every 4th month (chronological) is held out for validation

train_files = sorted(os.listdir(TRAIN_IMG))
unparsed = [f for f in train_files if date_key(f) is None]
print('unparsable train names:', unparsed[:5], '...' if len(unparsed) > 5 else '')

months = sorted({date_key(f) for f in train_files if date_key(f)})
val_months = {m for i, m in enumerate(months) if i % VAL_EVERY == VAL_EVERY - 1}
print(f'{len(months)} months in train, {len(val_months)} held out for validation')
print('validation months:', sorted(val_months))

unparsable train names: [] 
118 months in train, 29 held out for validation
validation months: ['201104', '201108', '201112', '201204', '201208', '201212', '201304', '201308', '201312', '201404', '201408', '201412', '201504', '201508', '201512', '201604', '201608', '201612', '201704', '201708', '201801', '201807', '201811', '201907', '202006', '202101', '202107', '202112', '202206']


In [38]:
YDS = f'{WORK}/filament_yolo'
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    os.makedirs(f'{YDS}/{sub}', exist_ok=True)

ann_by_img = defaultdict(list)
for a in anns:
    ann_by_img[a['image_id']].append(a)

stats = Counter()
for im in imgs:
    file_name = im['file_name']
    split = 'val' if date_key(file_name) in val_months else 'train'
    src = os.path.join(TRAIN_IMG, file_name)
    if not os.path.exists(src):
        stats['missing_image'] += 1
        continue

    # COCO id is unique (annotator batch + file name); use it as the YOLO sample name.
    stem = f"{im['id']}"
    link = f'{YDS}/images/{split}/{stem}.jpeg'
    if not os.path.exists(link):
        os.symlink(src, link)

    W, H = im.get('width', 2048), im.get('height', 2048)
    lines = []
    for a in ann_by_img[im['id']]:
        seg = a['segmentation'][0] if a['segmentation'] else []
        if len(seg) < 6:                      # fewer than 3 points: not a polygon
            stats['skip_degenerate'] += 1
            continue
        pts = np.asarray(seg, dtype=np.float32).reshape(-1, 2)
        pts[:, 0] = np.clip(pts[:, 0] / W, 0.0, 1.0)
        pts[:, 1] = np.clip(pts[:, 1] / H, 0.0, 1.0)
        lines.append('0 ' + ' '.join(f'{v:.6f}' for v in pts.reshape(-1)))

    with open(f'{YDS}/labels/{split}/{stem}.txt', 'w') as fh:
        fh.write('\n'.join(lines))
    stats[f'{split}_images'] += 1
    stats[f'{split}_filaments'] += len(lines)

print(dict(stats))

{'train_images': 872, 'train_filaments': 6105, 'val_images': 282, 'val_filaments': 2094}


In [39]:
with open(f'{YDS}/dataset.yaml', 'w') as fh:
    fh.write(f"path: {YDS}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n")
print(open(f'{YDS}/dataset.yaml').read())

for split in ('train', 'val'):
    lab = f'{YDS}/labels/{split}'
    files = os.listdir(lab)
    empty = sum(os.path.getsize(os.path.join(lab, f)) == 0 for f in files)
    print(f'{split}: {len(os.listdir(f"{YDS}/images/{split}"))} images, '
          f'{len(files)} label files, {empty} without any filament')

path: /kaggle/working/filament_yolo
train: images/train
val: images/val
names:
  0: filament

train: 872 images, 872 label files, 0 without any filament
val: 282 images, 282 label files, 0 without any filament


## Cell 4 — train

`imgsz` is the ablation worth running first: filaments are thin, and downscaling a 2048x2048 frame to
640 can erase them. Run the baseline at 1024, then repeat at 640 with everything else fixed.

Keep `cache=False` unless RAM allows ~9 GB (700 x 2048 x 2048 x 3 bytes decoded).

In [40]:
from ultralytics import YOLO
import os, shutil

MODEL   = 'yolo11n-seg.pt'     # alternative for the model ablation: 'yolo26n-seg.pt'
IMGSZ   = 1024                 # 1024 beats 640 on PQ (0.295 vs 0.205)
EPOCHS  = 30                   # 15 epochs was still improving; 30 is the fuller baseline
BATCH   = 4                    # lower to 2 if the T4 runs out of memory
CONF    = 0.10                 # best of the 0.10/0.25/0.50 sweep (Cell 6 still sweeps)
base    = MODEL.split('.')[0]
RUN     = f'{WORK}/runs/seg_{base}_{IMGSZ}'
WEIGHTS = f'{RUN}/weights/best.pt'
RESULTS = f'{RUN}/results.csv'

# Skip training only when a FULL run is on disk: ultralytics writes best.pt after the first
# epoch, so a partially trained run must not be mistaken for a finished one.
done_epochs = 0
if os.path.exists(RESULTS):
    with open(RESULTS) as fh:
        done_epochs = sum(1 for _ in fh) - 1
print(f'epochs already completed: {done_epochs}')

if done_epochs >= EPOCHS and os.path.exists(WEIGHTS):
    print('full run found, skipping training')
else:
    if os.path.exists(RUN):
        shutil.rmtree(RUN)     # drop the partial run so results.csv starts clean
    YOLO(MODEL).train(
        data=f'{YDS}/dataset.yaml',
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=0,                  # single T4; use device=[0, 1] to train on both cards
        workers=0,                 # single-process loading: the multi-worker loader stalled
        cache='ram',               # read every image once and never touch the disk again
        project=f'{WORK}/runs', name=f'seg_{base}_{IMGSZ}',
        exist_ok=True, seed=0, plots=True,
    )

model = YOLO(WEIGHTS)          # always evaluate the best checkpoint
print('weights:', WEIGHTS)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
epochs already completed: 0
Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/filament_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/30      3.46G      1.802      1.946       2.09       1.21          0         20       1024: 100% ━━━━━━━━━━━━ 218/218 4.2it/s 51.9s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 4.6it/s 7.8s0.2s
                   all        282       2094      0.498      0.506       0.46      0.203      0.515      0.489      0.451      0.142

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/30      3.46G      1.791      1.893      1.857      1.188          0         22       1024: 100% ━━━━━━━━━━━━ 218/218 4.2it/s 52.2s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 4.5it/s 7.9s0.2s
                   all        282       2094      0.458      0.516      0.451      0.206      0.444      0.497       0.42   

## Cell 5 — predict the test set and write `submission.csv`

Three rules taken from the official "Submission File" section:
1. masks must not overlap inside one image -> walk predictions from the highest confidence down and
   subtract already-claimed pixels;
2. RLE must be built at the original 2048x2048 resolution (`retina_masks=True`);
3. `filament_id` = `<image stem>_<n>`, `segmentation_rle` = RLE counts only, no size, no quotes.

In [41]:
from ultralytics import YOLO
from pycocotools import mask as maskUtils

WEIGHTS   = f'{RUN}/weights/best.pt'
MIN_PIX   = 16        # drop specks smaller than this many pixels
MAX_DET   = 100       # cap candidates: extras past this are mostly FP and dominate the runtime
model     = YOLO(WEIGHTS)

rows, n_empty, n_raw = [], 0, 0
test_files = sorted(os.listdir(TEST_IMG))
for f in test_files:
    stem = os.path.splitext(f)[0]
    res = model.predict(os.path.join(TEST_IMG, f), imgsz=IMGSZ, conf=CONF, device=0,
                        verbose=False, retina_masks=True, max_det=MAX_DET)[0]
    h, w = res.orig_shape
    if res.masks is None:
        n_empty += 1
        continue

    masks = res.masks.data.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()
    n_raw += len(confs)
    order = np.argsort(-confs)
    occupied = np.zeros((h, w), dtype=bool)
    k = 0
    for i in order:
        m = masks[i] > 0.5
        if m.shape != (h, w):
            m = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        m &= ~occupied               # rule 1: no overlapping masks
        if m.sum() < MIN_PIX:
            continue
        occupied |= m
        k += 1
        rle = maskUtils.encode(np.asfortranarray(m.astype(np.uint8)))
        rows.append((f'{stem}_{k}', rle['counts'].decode('utf-8')))

sub = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
sub.to_csv(f'{WORK}/submission.csv', index=False)
print('rows:', len(sub), '| images with no prediction:', n_empty, '/', len(test_files))
print('raw candidates kept for encoding:', n_raw)
print('columns:', list(sub.columns))
print(sub.head(3).to_string())


rows: 2205 | images with no prediction: 2 / 180
raw candidates kept for encoding: 2540
columns: ['filament_id', 'segmentation_rle']
          filament_id                                                                                                                                                                                                                                                                                                                                                                                                                                             segmentation_rle
0  20110120105534Ch_1                                                                                                                                                                                                                                                                                             ki\Q2<bo12O1O000010ON1O2N1O2N101O1O2L3N2N2O000010000N2N3M2N2OO2F:MAVOnQNk0Rn1[OaQNl0`n1:O01O1N2

## Cell 6 — local validation: mean Dice and PQ

Dice on the union of all filaments matches what the leaderboard reports; PQ is the metric the final
rubric ranks by, so both are computed here. Filament matching is greedy one-to-one at IoU > 0.5,
exactly as in the PQ definition.

In [42]:
from pycocotools import mask as maskUtils


def poly_to_rle(seg, h, w):
    return maskUtils.merge(maskUtils.frPyObjects([seg], h, w))


def mask_to_rle(m):
    return maskUtils.encode(np.asfortranarray(m.astype(np.uint8)))


# Ground truth as RLE; pycocotools does the GT x prediction pairing in C.
val_entries = [im for im in imgs if date_key(im['file_name']) in val_months]
gt_rle_by_file = defaultdict(list)
for im in val_entries:
    for a in ann_by_img[im['id']]:
        if a['segmentation']:
            gt_rle_by_file[im['file_name']].append(poly_to_rle(a['segmentation'][0], im['height'], im['width']))

# Predict each validation image ONCE at a low threshold and keep the masks, so the whole
# confidence sweep costs one prediction pass instead of one pass per threshold.
COLLECT_CONF = 0.05
CONF_SWEEP = [0.10, 0.25, 0.50]
pred_by_file = {}
for f in sorted(gt_rle_by_file):
    res = model.predict(os.path.join(TRAIN_IMG, f), imgsz=IMGSZ, conf=COLLECT_CONF,
                        device=0, verbose=False, retina_masks=True, max_det=100)[0]
    if res.masks is None:
        pred_by_file[f] = ([], np.zeros(0, dtype=np.float32))
        continue
    rles = [mask_to_rle(m) for m in (res.masks.data.cpu().numpy() > 0.5)]
    pred_by_file[f] = (rles, res.boxes.conf.cpu().numpy())


def evaluate(conf):
    dices, image_pq = [], []
    T = Counter()
    for f, (rles, confs) in pred_by_file.items():
        pr = [rles[i] for i in range(len(confs)) if confs[i] >= conf]
        gt_rles = gt_rle_by_file[f]
        h, w = gt_rles[0]['size']
        gt_m = maskUtils.decode(maskUtils.merge(gt_rles)).astype(bool)
        pr_m = maskUtils.decode(maskUtils.merge(pr)).astype(bool) if pr else np.zeros((h, w), bool)
        dices.append(2.0 * (gt_m & pr_m).sum() / max(gt_m.sum() + pr_m.sum(), 1))
        tp, sum_iou = 0, 0.0
        if gt_rles and pr:
            iou = maskUtils.iou(pr, gt_rles, [0] * len(gt_rles)).T
            while iou.size:
                ii, jj = np.unravel_index(int(np.argmax(iou)), iou.shape)
                if iou[ii, jj] <= 0.5:
                    break
                tp += 1
                sum_iou += float(iou[ii, jj])
                iou[ii, :] = 0.0
                iou[:, jj] = 0.0
        fp, fn = len(pr) - tp, len(gt_rles) - tp
        image_pq.append(sum_iou / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) else 0.0)
        T['tp'] += tp; T['fp'] += fp; T['fn'] += fn; T['sum_iou'] += sum_iou
    return np.mean(dices), np.mean(image_pq), T['sum_iou'] / (T['tp'] + 0.5 * T['fp'] + 0.5 * T['fn']), T


print(f'validation images: {len(pred_by_file)}  (predicted once at conf={COLLECT_CONF})')
print(f"{'conf':>6} {'meanDice':>9} {'imgPQ':>8} {'globalPQ':>9} {'tp':>6} {'fp':>6} {'fn':>6}")
for c in CONF_SWEEP:
    dice, ipq, gpq, T = evaluate(c)
    print(f'{c:>6.2f} {dice:>9.4f} {ipq:>8.4f} {gpq:>9.4f} {T["tp"]:>6} {T["fp"]:>6} {T["fn"]:>6}')


validation images: 174  (predicted once at conf=0.05)
  conf  meanDice    imgPQ  globalPQ     tp     fp     fn
  0.10    0.6396   0.2869    0.2816   1030   1490   1064
  0.25    0.6397   0.2884    0.2845    767    575   1327
  0.50    0.5192   0.2297    0.2216    462    140   1632


## Cell 7 — GT-protocol sensitivity (four ground-truth protocols)


In [43]:
# Cell 7 -- fixed evaluation: de-duplicate the validation ground truth.
# Supersedes Cell 6. One prediction pass, four GT protocols:
#   P0 all annotator copies merged (what Cell 6 did) / P1 first copy / P2 last copy / P3 random x5
import random
from collections import Counter, defaultdict
import numpy as np
from pycocotools import mask as maskUtils


def poly_to_rle(seg, h, w):
    return maskUtils.merge(maskUtils.frPyObjects([seg], h, w))


def mask_to_rle(m):
    return maskUtils.encode(np.asfortranarray(m.astype(np.uint8)))


MAX_DET_EVAL = 100

val_entries = [im for im in imgs if date_key(im['file_name']) in val_months]
copies_by_file = defaultdict(list)
for im in val_entries:
    copies_by_file[im['file_name']].append(im)
n_entries = len(val_entries)
n_files = len(copies_by_file)
n_gt_ann = sum(len(ann_by_img[im['id']]) for im in val_entries)
print('val COCO entries   :', n_entries)
print('distinct files     :', n_files)
print('files with >1 copy :', sum(1 for v in copies_by_file.values() if len(v) > 1))
print('val annotations    :', n_gt_ann)
print('copies per file    :', dict(sorted(Counter(len(v) for v in copies_by_file.values()).items())))

pred_by_file = {}
for f in sorted(copies_by_file):
    res = model.predict(os.path.join(TRAIN_IMG, f), imgsz=IMGSZ, conf=CONF,
                        device=0, verbose=False, retina_masks=True, max_det=MAX_DET_EVAL)[0]
    pred_by_file[f] = [] if res.masks is None else [
        mask_to_rle(m) for m in (res.masks.data.cpu().numpy() > 0.5)]
print('predicted filaments:', sum(len(v) for v in pred_by_file.values()))


def build_gt(pick):
    out = defaultdict(list)
    for f, entries in copies_by_file.items():
        for im in (entries if pick is None else [pick(entries)]):
            for a in ann_by_img[im['id']]:
                if a['segmentation']:
                    out[f].append(poly_to_rle(a['segmentation'][0], im['height'], im['width']))
    return out


def evaluate(gt_by_file, tag, quiet=False):
    dices, image_pq, T = [], [], Counter()
    for f in sorted(copies_by_file):
        gt_rles = gt_by_file.get(f, [])
        pr_rles = pred_by_file[f]
        h = copies_by_file[f][0]['height']
        w = copies_by_file[f][0]['width']
        gt_m = maskUtils.decode(maskUtils.merge(gt_rles)).astype(bool) if gt_rles else np.zeros((h, w), bool)
        pr_m = maskUtils.decode(maskUtils.merge(pr_rles)).astype(bool) if pr_rles else np.zeros((h, w), bool)
        dices.append(2.0 * (gt_m & pr_m).sum() / max(gt_m.sum() + pr_m.sum(), 1))
        tp, sum_iou = 0, 0.0
        if gt_rles and pr_rles:
            iou = maskUtils.iou(pr_rles, gt_rles, [0] * len(gt_rles)).T
            while iou.size:
                ii, jj = np.unravel_index(int(np.argmax(iou)), iou.shape)
                if iou[ii, jj] <= 0.5:
                    break
                tp += 1
                sum_iou += float(iou[ii, jj])
                iou[ii, :] = 0.0
                iou[:, jj] = 0.0
        fp = len(pr_rles) - tp
        fn = len(gt_rles) - tp
        image_pq.append(sum_iou / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) else 0.0)
        T['tp'] += tp
        T['fp'] += fp
        T['fn'] += fn
        T['sum_iou'] += sum_iou
    gpq = T['sum_iou'] / (T['tp'] + 0.5 * T['fp'] + 0.5 * T['fn'])
    rec = dict(tag=tag, meanDice=float(np.mean(dices)), imagePQ=float(np.mean(image_pq)),
               globalPQ=float(gpq), tp=T['tp'], fp=T['fp'], fn=T['fn'], gt_inst=T['tp'] + T['fn'])
    if not quiet:
        print(f'{tag:<26} meanDice={rec["meanDice"]:.4f} imagePQ={rec["imagePQ"]:.4f} '
              f'globalPQ={rec["globalPQ"]:.4f} tp={rec["tp"]} fp={rec["fp"]} fn={rec["fn"]} GT={rec["gt_inst"]}')
    return rec


rows = []
rows.append(evaluate(build_gt(None), 'P0 all copies (Cell 6)'))
rows.append(evaluate(build_gt(lambda es: es[0]), 'P1 first annotator'))
rows.append(evaluate(build_gt(lambda es: es[-1]), 'P2 last annotator'))
rng = random.Random(0)
draws = [evaluate(build_gt(lambda es: rng.choice(es)), 'P3', quiet=True) for _ in range(5)]
g = np.array([d['globalPQ'] for d in draws])
dc = np.array([d['meanDice'] for d in draws])
print(f'{"P3 random x5":<26} globalPQ={g.mean():.4f} +/- {g.std():.4f}  meanDice={dc.mean():.4f} +/- {dc.std():.4f}')
p0 = rows[0]
p1 = rows[1]
print()
print('GT instances :', p0['gt_inst'], '->', p1['gt_inst'])
print('FN           :', p0['fn'], '->', p1['fn'])
print('global PQ    : %.4f -> %.4f  (%+.4f)' % (p0['globalPQ'], p1['globalPQ'], p1['globalPQ'] - p0['globalPQ']))
print('mean Dice    : %.4f -> %.4f  (%+.4f)' % (p0['meanDice'], p1['meanDice'], p1['meanDice'] - p0['meanDice']))
print()
print('| GT protocol | mean Dice | image-PQ | global-PQ | tp | fp | fn | GT instances |')
print('|---|---|---|---|---|---|---|---|')
for r in rows:
    print('| %s | %.4f | %.4f | %.4f | %d | %d | %d | %d |' % (
        r['tag'], r['meanDice'], r['imagePQ'], r['globalPQ'], r['tp'], r['fp'], r['fn'], r['gt_inst']))
print('| P3 random (mean of 5) | %.4f | - | %.4f | - | - | - | - |' % (dc.mean(), g.mean()))

val COCO entries   : 282
distinct files     : 174
files with >1 copy : 71
val annotations    : 2094
copies per file    : {1: 103, 2: 34, 3: 37}
predicted filaments: 2520
P0 all copies (Cell 6)     meanDice=0.6396 imagePQ=0.2869 globalPQ=0.2816 tp=1030 fp=1490 fn=1064 GT=2094
P1 first annotator         meanDice=0.6053 imagePQ=0.2710 globalPQ=0.2622 tp=809 fp=1711 fn=532 GT=1341
P2 last annotator          meanDice=0.6161 imagePQ=0.2854 globalPQ=0.2735 tp=842 fp=1678 fn=475 GT=1317
P3 random x5               globalPQ=0.2663 +/- 0.0021  meanDice=0.6090 +/- 0.0035

GT instances : 2094 -> 1341
FN           : 1064 -> 532
global PQ    : 0.2816 -> 0.2622  (-0.0194)
mean Dice    : 0.6396 -> 0.6053  (-0.0343)

| GT protocol | mean Dice | image-PQ | global-PQ | tp | fp | fn | GT instances |
|---|---|---|---|---|---|---|---|
| P0 all copies (Cell 6) | 0.6396 | 0.2869 | 0.2816 | 1030 | 1490 | 1064 | 2094 |
| P1 first annotator | 0.6053 | 0.2710 | 0.2622 | 809 | 1711 | 532 | 1341 |
| P2 last annotato

## Cell 8 — regenerate the submission at conf = 0.25


In [45]:
# Cell 8 -- re-generate the test submission at conf=0.25.
# Rationale (validation, same weights): conf 0.25 vs 0.10 -> fp 1490 to 575,
# global PQ 0.2816 to 0.2845, Dice unchanged (0.6396 vs 0.6397).
# Keeps the previous conf=0.10 file as a backup.
import shutil

OLD = f'{WORK}/submission_conf010.csv'
if os.path.exists(f'{WORK}/submission.csv') and not os.path.exists(OLD):
    shutil.copy(f'{WORK}/submission.csv', OLD)
    print('backed up previous submission ->', OLD)

CONF_SUB = 0.25
rows, n_empty, n_raw = [], 0, 0
test_files = sorted(os.listdir(TEST_IMG))
for f in test_files:
    stem = os.path.splitext(f)[0]
    res = model.predict(os.path.join(TEST_IMG, f), imgsz=IMGSZ, conf=CONF_SUB, device=0,
                        verbose=False, retina_masks=True, max_det=MAX_DET)[0]
    h, w = res.orig_shape
    if res.masks is None:
        n_empty += 1
        continue
    masks = res.masks.data.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()
    n_raw += len(confs)
    occupied = np.zeros((h, w), dtype=bool)
    k = 0
    for i in np.argsort(-confs):
        m = masks[i] > 0.5
        if m.shape != (h, w):
            m = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
        m &= ~occupied
        if m.sum() < MIN_PIX:
            continue
        occupied |= m
        k += 1
        rle = maskUtils.encode(np.asfortranarray(m.astype(np.uint8)))
        rows.append((f'{stem}_{k}', rle['counts'].decode('utf-8')))

sub = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
sub.to_csv(f'{WORK}/submission.csv', index=False)
sub.to_csv(f'{WORK}/submission_conf025.csv', index=False)
print('rows:', len(sub), '| images with no prediction:', n_empty, '/', len(test_files))
print('raw candidates kept for encoding:', n_raw)
print('written:', f'{WORK}/submission.csv', '+ submission_conf025.csv')

rows: 1286 | images with no prediction: 2 / 180
raw candidates kept for encoding: 1365
written: /kaggle/working/submission.csv + submission_conf025.csv


## Cell 9 — instance-level distributions (the rubric's three items)


In [46]:
# Cell 9 -- the three instance-level distributions the rubric scores.
#
#   (1) per-image Dice distribution
#   (2) matched-pair IoU distribution
#   (3) one-to-many / many-to-one relations  (fragmentation vs merging)
#
# The competition rubric lists these alongside mean Dice and PQ, and the official
# self-evaluation notebook computes them with `get_overlap_matrices`,
# `fp_count_hit`, `fn_count_hit`. This cell recomputes the same quantities in the
# vocabulary the report already uses.
#
# PREREQUISITE: must run AFTER Cell 7 in the same session -- it reuses
# `pred_by_file`, `copies_by_file` and `build_gt` from there.
# GT protocol = P0 (union of all annotator copies), i.e. the headline protocol.

import numpy as np

THR = 0.5          # same IoU threshold as PQ
NBINS = 10

gt_by_file = build_gt(None)

dices = []
pair_iou = []
many_to_one = Counter()   # for each GT instance: how many predictions claim it  -> merging
one_to_many = Counter()   # for each prediction:  how many GT instances it claims -> fragmentation
tp = fp = fn = 0

for f in sorted(copies_by_file):
    gt_rles = gt_by_file.get(f, [])
    pr_rles = pred_by_file[f]
    h = copies_by_file[f][0]['height']
    w = copies_by_file[f][0]['width']
    gt_m = maskUtils.decode(maskUtils.merge(gt_rles)).astype(bool) if gt_rles else np.zeros((h, w), bool)
    pr_m = maskUtils.decode(maskUtils.merge(pr_rles)).astype(bool) if pr_rles else np.zeros((h, w), bool)
    dices.append(2.0 * (gt_m & pr_m).sum() / max(gt_m.sum() + pr_m.sum(), 1))

    t = 0
    if gt_rles and pr_rles:
        iou = maskUtils.iou(pr_rles, gt_rles, [0] * len(gt_rles))   # (n_pred, n_gt)
        hit = iou > THR
        for j in range(len(gt_rles)):
            many_to_one[int(hit[:, j].sum())] += 1
        for i in range(len(pr_rles)):
            one_to_many[int(hit[i, :].sum())] += 1
        m = iou.T.copy()
        while m.size:
            a, b = np.unravel_index(int(np.argmax(m)), m.shape)
            if m[a, b] <= THR:
                break
            t += 1
            pair_iou.append(float(m[a, b]))
            m[a, :] = 0.0
            m[:, b] = 0.0
    tp += t
    fp += len(pr_rles) - t
    fn += len(gt_rles) - t

dices = np.asarray(dices, dtype=float)
pair_iou = np.asarray(pair_iou, dtype=float)


def hist_lines(vals, lo, hi, nbins):
    edges = np.linspace(lo, hi, nbins + 1)
    cnt, _ = np.histogram(vals, bins=edges)
    out = []
    for i in range(nbins):
        bar = '#' * int(round(60.0 * cnt[i] / max(cnt.max(), 1)))
        out.append('  %4.2f-%4.2f | %5d %s' % (edges[i], edges[i + 1], cnt[i], bar))
    return out


def pct(vals, q):
    return float(np.percentile(vals, q))


print('=== (1) per-image Dice distribution  (n=%d images) ===' % len(dices))
print('  mean %.4f | median %.4f | mean of top/bottom decile %.4f / %.4f'
      % (dices.mean(), np.median(dices), pct(dices, 90), pct(dices, 10)))
print('  p10 %.4f  p25 %.4f  p50 %.4f  p75 %.4f  p90 %.4f'
      % (pct(dices, 10), pct(dices, 25), pct(dices, 50), pct(dices, 75), pct(dices, 90)))
print('  images with Dice = 0 (total miss): %d (%.1f%%)'
      % (int((dices == 0).sum()), 100.0 * (dices == 0).mean()))
for line in hist_lines(dices, 0.0, 1.0, NBINS):
    print(line)

print()
print('=== (2) matched-pair IoU distribution  (n=%d matched pairs, IoU > %.1f) ===' % (len(pair_iou), THR))
print('  mean %.4f | median %.4f | p10 %.4f | p90 %.4f'
      % (pair_iou.mean(), np.median(pair_iou), pct(pair_iou, 10), pct(pair_iou, 90)))
for line in hist_lines(pair_iou, THR, 1.0, NBINS):
    print(line)
print('  NOTE: every pair here is already above %.1f, so this measures how *well* the' % THR)
print('        matched masks agree, not how many are missed.')

print()
print('=== (3) relations between predictions and GT ===')
print('  tp=%d  fp=%d  fn=%d   (global PQ = %.4f over the union protocol)'
      % (tp, fp, fn, sum(pair_iou) / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) else 0.0))
keys = sorted(set(list(many_to_one) + list(one_to_many)))
print('  %-6s %-14s %-16s' % ('n', 'GT claimed by', 'prediction claims'))
print('  %-6s %-14s %-16s' % ('', 'n predictions', 'n GT instances'))
for k in keys:
    label = {0: '0 (missed)', 1: '1 (clean)'}.get(k, '%d (merged)' % k)
    print('  %-6d %-14d %-16d' % (k, many_to_one.get(k, 0), one_to_many.get(k, 0)))
print('  fragmentation: predictions claiming >1 GT = %d (%.1f%% of all predictions)'
      % (sum(v for k, v in one_to_many.items() if k > 1),
         100.0 * sum(v for k, v in one_to_many.items() if k > 1) / max(sum(one_to_many.values()), 1)))
print('  merging:       GT claimed by >1 prediction  = %d (%.1f%% of all GT)'
      % (sum(v for k, v in many_to_one.items() if k > 1),
         100.0 * sum(v for k, v in many_to_one.items() if k > 1) / max(sum(many_to_one.values()), 1)))

# ---- figure for the report (2 panels, no styling magic) ----------------------
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].hist(dices, bins=20, range=(0, 1), color='#3b6fb6', edgecolor='white')
    ax[0].set_title('Per-image Dice (n=%d)' % len(dices))
    ax[0].set_xlabel('Dice'); ax[0].set_ylabel('images')
    ax[1].hist(pair_iou, bins=20, range=(THR, 1), color='#c0504d', edgecolor='white')
    ax[1].set_title('Matched-pair IoU (n=%d, all > %.1f)' % (len(pair_iou), THR))
    ax[1].set_xlabel('IoU'); ax[1].set_ylabel('matched pairs')
    for a in ax:
        a.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    OUT = f'{WORK}/fig_distributions.png'
    fig.savefig(OUT, dpi=160)
    fig.savefig(f'{WORK}/fig_distributions.pdf')
    print()
    print('figure written:', OUT, '+ .pdf')
except Exception as e:
    print()
    print('figure skipped (%s)' % e)

=== (1) per-image Dice distribution  (n=174 images) ===
  mean 0.6396 | median 0.6628 | mean of top/bottom decile 0.7586 / 0.4905
  p10 0.4905  p25 0.5853  p50 0.6628  p75 0.7222  p90 0.7586
  images with Dice = 0 (total miss): 1 (0.6%)
  0.00-0.10 |     1 #
  0.10-0.20 |     1 #
  0.20-0.30 |     0 
  0.30-0.40 |     7 #######
  0.40-0.50 |    10 ##########
  0.50-0.60 |    32 ###############################
  0.60-0.70 |    62 ############################################################
  0.70-0.80 |    56 ######################################################
  0.80-0.90 |     5 #####
  0.90-1.00 |     0 

=== (2) matched-pair IoU distribution  (n=1030 matched pairs, IoU > 0.5) ===
  mean 0.6307 | median 0.6217 | p10 0.5279 | p90 0.7429
  0.50-0.55 |   190 ####################################################
  0.55-0.60 |   221 ############################################################
  0.60-0.65 |   213 ##########################################################
  0.65-0.70 |   1

## Next steps (ablation plan for the report)

Run one change at a time, keeping everything else fixed, and record Dice + PQ for each:
1. `IMGSZ` 1024 vs 640 — the ablation to run first, filaments are thin;
2. `CONF` 0.1 / 0.25 / 0.5 — watch how it trades false positives against false negatives;
3. `MODEL` yolo11n-seg vs yolo26n-seg;
4. optional: `VAL_EVERY` (3/4/5) to show how sensitive the score is to the validation split.

Failed cases are worth keeping: the report asks for a few failures with one sentence of explanation
each (fragmented filaments and merged neighbouring dark regions are the expected failure modes).